# N03 · DDP 与 Collectives：分布式训练为什么会 hang？


> 学习方式建议：先读“心智模型”，再运行代码实验，最后做练习题。每道题都附答案解析，不是为了考倒你，而是为了暴露最常见的模棱两可点。  
> 本 notebook 只做概念和可复现小实验；真实工程闭环请回到对应 lab 运行 `make smoke M=...` 并查看 `runs/.../metrics.jsonl`、日志和报告。


## 本节要解决的问题

分布式训练最折磨人的问题之一是：程序不报错，只是卡住。要理解 hang，必须先理解 collective：所有 rank 按相同顺序参与同一个通信操作。一个 rank 少走一步，其他 rank 就可能一直等。

本节会把 rank/world size、process group、all-reduce、all-gather、broadcast、reduce-scatter、DDP 梯度同步串起来。


## 学习地图与版本说明（截至 2026-04-30）

本节从 collective 的同步契约入手：所有 rank 必须以兼容的张量形状、dtype 和调用顺序进入同一类通信。只要有一个 rank 分支不同、提前异常、少跑一次 backward 或进入不同 collective，其他 rank 就可能表现为“没有报错但一直等”。

版本上，本教程优先参考 PyTorch stable 的 `torch.distributed`、`DistributedDataParallel` 和 `torchrun` 文档。当前文档仍建议 CUDA GPU 训练优先使用 NCCL 后端；DDP 是同步数据并行封装，核心是每个进程持有模型副本并在 backward 中同步梯度。注意：DDP 解决吞吐扩展问题，不等于自动解决单卡放不下的问题；单模型显存仍需要 FSDP/ZeRO、TP、PP 或 checkpointing 等技术。

学完本节，你应该能画出一次 DDP step：每个 rank 读不同数据分片、forward/backward、梯度 bucket all-reduce、optimizer step；也能说清楚 `RANK/WORLD_SIZE/LOCAL_RANK` 的来源，以及为什么分布式 bug 优先用 2 rank、极小 batch、固定随机种子复现。


## 1. rank/world size/local rank 是什么？

- `WORLD_SIZE`：这次分布式作业总共有多少个进程。
- `RANK`：当前进程在全局中的编号，范围 `[0, WORLD_SIZE-1]`。
- `LOCAL_RANK`：当前进程在本机的编号，常用来绑定本机 GPU。

单机 8 卡时通常 `WORLD_SIZE=8`，每个进程一个 GPU。多机时 `RANK` 跨机器全局唯一，`LOCAL_RANK` 只在本机内有意义。

常见错误：用 `RANK` 直接当 CUDA device id。在多机时 rank 8 可能在第二台机器，但本机只有 `cuda:0..7`。


In [ ]:
def rank_table(nodes=2, gpus_per_node=4):
    rows = []
    for node in range(nodes):
        for local_rank in range(gpus_per_node):
            rank = node * gpus_per_node + local_rank
            rows.append({"node": node, "RANK": rank, "LOCAL_RANK": local_rank, "cuda_device": f"cuda:{local_rank}"})
    return pd.DataFrame(rows)

import pandas as pd
display(rank_table())


## 2. Collective 的本质：所有人都要在同一个路口集合

想象 4 个 rank 约好：

1. 先在 A 路口 all-reduce。
2. 再在 B 路口 all-gather。

如果 rank 2 因为条件分支跳过 A，直接去了 B，其它 rank 会在 A 等它；rank 2 会在 B 等别人。于是大家都“没有报错”，但系统死锁。

这就是为什么分布式代码中 **collective 的调用顺序必须跨 rank 一致**。


In [ ]:
# 用纯 Python 模拟 collective 的顺序一致性。
rank_ops_ok = {
    0: ["all_reduce", "all_gather", "broadcast"],
    1: ["all_reduce", "all_gather", "broadcast"],
    2: ["all_reduce", "all_gather", "broadcast"],
}
rank_ops_bad = {
    0: ["all_reduce", "all_gather", "broadcast"],
    1: ["all_reduce", "broadcast"],
    2: ["all_reduce", "all_gather", "broadcast"],
}

def check_collective_order(rank_ops):
    max_len = max(len(v) for v in rank_ops.values())
    problems = []
    for step in range(max_len):
        ops = {rank: ops[step] if step < len(ops) else "<missing>" for rank, ops in rank_ops.items()}
        if len(set(ops.values())) != 1:
            problems.append((step, ops))
    return problems

print("正常顺序问题：", check_collective_order(rank_ops_ok))
print("错误顺序问题：", check_collective_order(rank_ops_bad))


## 3. all-reduce、all-gather、reduce-scatter、broadcast 怎么区分？

- **all-reduce**：每个 rank 有一个 tensor，做 sum/mean/max 后，每个 rank 都拿到相同结果。DDP 梯度同步最典型。
- **all-gather**：每个 rank 有一片数据，最后每个 rank 都收集到所有片。
- **reduce-scatter**：先 reduce，再把结果切片分给各 rank。FSDP/ZeRO 中常见。
- **broadcast**：一个源 rank 把数据发给所有 rank。模型初始化或参数同步常见。

面试痛点：不要把 all-reduce 和 all-gather 混为一谈。all-reduce 改变的是“数值聚合”，all-gather 改变的是“数据拼接”。


In [ ]:
import numpy as np

ranks = {0: np.array([1, 1]), 1: np.array([2, 2]), 2: np.array([3, 3])}
all_reduce_sum = sum(ranks.values())
all_gather = np.concatenate([ranks[i] for i in sorted(ranks)])
reduce_scatter_chunks = np.array_split(all_reduce_sum, len(ranks))
broadcast_from_0 = {rank: ranks[0].copy() for rank in ranks}

print("all_reduce_sum 每个 rank 得到:", all_reduce_sum)
print("all_gather 每个 rank 得到:", all_gather)
print("reduce_scatter 各 rank 得到:", reduce_scatter_chunks)
print("broadcast from rank0:", broadcast_from_0)


## 4. DDP 做了什么？

`DistributedDataParallel` 的核心是：每个 rank 有一份模型副本，处理不同 mini-batch；backward 时对梯度做 all-reduce，让每个 rank 的梯度平均一致，然后各自 optimizer step。

因此 DDP 的语义接近“大 batch 数据并行”：

```text
global_batch = micro_batch_per_rank × world_size × gradient_accumulation
```

常见误区：DDP 不会自动让模型变小；每张卡仍有完整模型权重。它主要扩大数据吞吐，而不是解决单卡放不下模型的问题。放不下模型要看 FSDP/ZeRO/TP/PP。


## 5. 分布式 hang 的最小排查清单

1. 每个 rank 启动时打印 `RANK/LOCAL_RANK/WORLD_SIZE/hostname/device`。
2. 确认所有 rank 都进入同一个训练 step。
3. 在关键 collective 前后打日志：`before all_reduce step=... rank=...`。
4. 设 timeout，避免无限等。
5. 检查某些 rank 是否数据提前耗尽，例如 DistributedSampler 没有 drop_last 或 epoch 设置不一致。
6. 检查某些 rank 是否 OOM/异常退出，但其他 rank 还在等 collective。

与本课程连接：L02 的 `collectives_demo.py` 和 `ddp_toy_train.py` 就是最小复现工具。


## 6. 企业面试/工程判断痛点题（带答案）

### 题 1：DDP 和 DataParallel 最大的工程差异是什么？

**答案解析：** DDP 是多进程，每个进程通常绑定一个 GPU，梯度通过 distributed backend 同步；DataParallel 是单进程多线程/多设备 scatter-gather，通常扩展性和性能较差。生产分布式训练优先 DDP/FSDP。

### 题 2：DDP 能解决 70B 模型单卡放不下的问题吗？

**答案解析：** 不能。DDP 每张卡仍保存完整模型副本。要解决模型放不下，需要 FSDP/ZeRO、tensor parallel、pipeline parallel、CPU/NVMe offload 等。

### 题 3：为什么一个 rank 少调用一次 all-reduce 会导致 hang？

**答案解析：** collective 要求同一 process group 内所有 rank 以一致顺序参与。少调用的 rank 去了下一个 collective 或退出，其它 rank 会在当前 collective 等它。

### 题 4：多机训练中为什么不能用 `RANK` 作为 CUDA device id？

**答案解析：** 全局 RANK 可能大于本机 GPU 数。应该用 `LOCAL_RANK` 绑定本机设备，例如 `torch.cuda.set_device(local_rank)`。

### 题 5：如果 DDP loss 不下降，能直接说 all-reduce 坏了吗？

**答案解析：** 不能。先检查数据、学习率、loss scale、梯度是否为 NaN、模型是否真的在 train mode。all-reduce 问题通常表现为 hang、报错、不同 rank 参数漂移，而不是唯一导致 loss 不下降。


## 7. 小结

分布式训练的第一性原理：**多进程必须在通信顺序、tensor shape、设备绑定和数据步数上保持一致**。DDP 只是把这一套封装得更好，但 debug 时仍要回到 rank/world size/collective。

下一步：运行 `torchrun --nproc_per_node=2 labs/l05_distributed_primitives/scripts/collectives_demo.py`，观察真实 rank 输出。


## 参考资料

- PyTorch Distributed package: https://docs.pytorch.org/docs/stable/distributed.html
- PyTorch DistributedDataParallel: https://docs.pytorch.org/docs/stable/generated/torch.nn.parallel.DistributedDataParallel.html
- torchrun / Elastic Launch: https://docs.pytorch.org/docs/stable/elastic/run.html
